# Optimizations vs baseline — Opus CLI

Each CLI optimization (`opt1-batch` … `opt6-stable-output`) and the **baseline** (plain `cli`) is shown in two
variants, side by side:

- **no-skills** — the CLI build alone, no skill bundle.
- **+skills** — the same build with its tuned *optimized* skill bundle.

The baseline arm is `1_hw-full-cli-sdk-skills-opus` restricted to its plain `cli` / no-skills runs (no-skills) and
`11_hw-cli-skills-optimized-opus` = plain `cli` + optimized skills (+skills). Everything is Opus / CLI, so this
isolates the effect of the optimization **and** of adding skills on top of it.

For each arm we average three metrics across the tasks that BOTH variants ran *valid* (paired, so a missing/failed
task never skews the mean):

- **local_time_s** — wall time spent locally (lower is better)
- **cost_usd** — dollar cost of the run (lower is better)
- **pass rate** — `asserts_passed / total_asserts` (higher is better)

Only `valid` runs are counted. Dashed line on the charts = the absolute baseline (no-skills plain `cli`).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

df = pd.read_csv("results.csv")
for c in ["asserts_passed", "total_asserts", "local_time_s", "cost_usd"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["valid"] = df["valid"].astype(str).str.lower() == "true"
df["pass_rate"] = df["asserts_passed"] / df["total_asserts"]
valid = df[df.valid].copy()

# Each "arm" = a CLI build (baseline = plain cli), shown WITHOUT its optimized skill
# bundle (no-skills) and WITH it (+skills). Format: (label, no-skills selector, +skills config).
# A selector is a config string, or (config, interface, skills) when a config has several variants.
ARMS = [
    ("baseline",           ("1_hw-full-cli-sdk-skills-opus", "cli", "none"), "11_hw-cli-skills-optimized-opus"),
    ("opt1-batch",         ("5_hw-cli-opt1-batch-opus", None, None),         "12_hw-cli-opt1-batch-skills-opt-opus"),
    ("opt2-session-reuse", ("6_hw-cli-opt2-session-reuse-opus", None, None), "13_hw-cli-opt2-session-reuse-skills-opt-opus"),
    ("opt3-compact-json",  ("7_hw-cli-opt3-compact-json-opus", None, None),  "14_hw-cli-opt3-compact-json-skills-opt-opus"),
    ("opt4-idempotent",    ("8_hw-cli-opt4-idempotent-opus", None, None),    "15_hw-cli-opt4-idempotent-skills-opt-opus"),
    ("opt5-quiet",         ("9_hw-cli-opt5-quiet-opus", None, None),         "16_hw-cli-opt5-quiet-skills-opt-opus"),
    ("opt6-stable-output", ("10_hw-cli-opt6-stable-output-opus", None, None),"17_hw-cli-opt6-stable-output-skills-opt-opus"),
]
METRICS = [
    ("local_time_s", "local time (s)",   "min"),
    ("cost_usd",     "cost (USD)",       "min"),
    ("pass_rate",    "assert pass rate", "max"),
]

def select(spec):
    """spec: a +skills config string, or a (config, interface, skills) tuple for no-skills."""
    if isinstance(spec, str):
        return valid[valid.config == spec]
    cfg, iface, sk = spec
    sub = valid[valid.config == cfg]
    if iface is not None:
        sub = sub[sub.interface == iface]
    if sk is not None:
        sub = sub[sub.skills == sk]
    return sub

def per_task_mean(sub, col):
    return sub.groupby("task")[col].mean()

# Per arm, per metric: (no-skills mean, +skills mean, n_paired_tasks), paired on the
# tasks both variants ran valid.
paired = {}
for label, ns_spec, sk_cfg in ARMS:
    ns_df, sk_df = select(ns_spec), select(sk_cfg)
    paired[label] = {}
    for col, _, _ in METRICS:
        ns, sk = per_task_mean(ns_df, col), per_task_mean(sk_df, col)
        common = [t for t in ns.index.intersection(sk.index) if pd.notna(ns[t]) and pd.notna(sk[t])]
        paired[label][col] = (ns[common].mean(), sk[common].mean(), len(common))


## Paired comparison table — no-skills vs +skills

`ns_*` = no-skills, `sk_*` = +skills, `skills_pct_*` = relative change from adding skills.

In [2]:
rows = []
for label, _, _ in ARMS:
    rec = {"arm": label}
    for col, _, _ in METRICS:
        ns, sk, n = paired[label][col]
        rec["n_tasks"] = n
        rec[f"ns_{col}"] = ns
        rec[f"sk_{col}"] = sk
        rec[f"skills_pct_{col}"] = (sk - ns) / ns * 100 if ns else float("nan")
    rows.append(rec)

summary = pd.DataFrame(rows).set_index("arm")
fmt = {}
for col, _, _ in METRICS:
    f = "{:.1f}" if col == "local_time_s" else ("{:.1%}" if col == "pass_rate" else "{:.4f}")
    fmt[f"ns_{col}"] = f
    fmt[f"sk_{col}"] = f
    fmt[f"skills_pct_{col}"] = "{:+.1f}%"
summary.style.format(fmt)


,n_tasks,ns_local_time_s,sk_local_time_s,skills_pct_local_time_s,ns_cost_usd,sk_cost_usd,skills_pct_cost_usd,ns_pass_rate,sk_pass_rate,skills_pct_pass_rate
arm,,,,,,,,,,
baseline,25,214.4,270.1,+26.0%,0.3169,0.3744,+18.1%,100.0%,98.7%,-1.3%
opt1-batch,24,249.4,286.9,+15.0%,0.3281,0.4203,+28.1%,98.6%,100.0%,+1.4%
opt2-session-reuse,24,213.7,236.0,+10.4%,0.3102,0.3419,+10.2%,97.2%,97.2%,+0.0%
opt3-compact-json,23,222.6,255.8,+14.9%,0.3055,0.3087,+1.0%,97.1%,96.4%,-0.7%
opt4-idempotent,24,234.7,225.4,-4.0%,0.3425,0.3235,-5.5%,100.0%,98.6%,-1.4%
opt5-quiet,24,222.4,252.1,+13.4%,0.3001,0.3556,+18.5%,98.6%,100.0%,+1.4%
opt6-stable-output,23,207.4,245.3,+18.3%,0.3001,0.3595,+19.8%,97.1%,100.0%,+3.0%


## Charts

Grouped bars per arm: no-skills (grey) vs +skills (blue). Dashed line = the absolute baseline (no-skills plain `cli`).

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
labels = [l for l, _, _ in ARMS]
x = np.arange(len(labels))
w = 0.38

for ax, (col, title, direction) in zip(axes, METRICS):
    ns_vals = [paired[l][col][0] for l in labels]
    sk_vals = [paired[l][col][1] for l in labels]
    b1 = ax.bar(x - w / 2, ns_vals, w, label="no-skills", color="#90a4ae")
    b2 = ax.bar(x + w / 2, sk_vals, w, label="+skills",  color="#1565c0")
    base_ref = paired["baseline"][col][0]  # absolute baseline = no-skills plain cli
    ax.axhline(base_ref, ls="--", lw=1, color="#555")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"{title}  ({'lower' if direction == 'min' else 'higher'} = better)")
    ax.set_ylabel(title)
    if col == "pass_rate":
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.legend(fontsize=8)
    for bars in (b1, b2):
        for b in bars:
            h = b.get_height()
            txt = f"{h:.0%}" if col == "pass_rate" else f"{h:.3g}"
            ax.annotate(txt, (b.get_x() + b.get_width() / 2, h),
                        ha="center", va="bottom", fontsize=7)

fig.suptitle("Optimizations vs baseline — no-skills vs +skills  (dashed = baseline no-skills)", y=1.02)
plt.tight_layout()
plt.show()


## Assertions: passed / total across the paired tasks

In [4]:
arows = []
for label, ns_spec, sk_cfg in ARMS:
    ns_df, sk_df = select(ns_spec), select(sk_cfg)
    ns_ok, ns_tot = per_task_mean(ns_df, "asserts_passed"), per_task_mean(ns_df, "total_asserts")
    sk_ok, sk_tot = per_task_mean(sk_df, "asserts_passed"), per_task_mean(sk_df, "total_asserts")
    common = ns_ok.index.intersection(sk_ok.index)
    rec = {
        "arm": label, "n_tasks": len(common),
        "ns_passed": ns_ok[common].sum(), "ns_total": ns_tot[common].sum(),
        "sk_passed": sk_ok[common].sum(), "sk_total": sk_tot[common].sum(),
    }
    rec["ns_rate"] = rec["ns_passed"] / rec["ns_total"] if rec["ns_total"] else float("nan")
    rec["sk_rate"] = rec["sk_passed"] / rec["sk_total"] if rec["sk_total"] else float("nan")
    arows.append(rec)

adf = pd.DataFrame(arows).set_index("arm")
adf.style.format({"ns_passed": "{:.0f}", "ns_total": "{:.0f}",
                  "sk_passed": "{:.0f}", "sk_total": "{:.0f}",
                  "ns_rate": "{:.1%}", "sk_rate": "{:.1%}"})


,n_tasks,ns_passed,ns_total,sk_passed,sk_total,ns_rate,sk_rate
arm,,,,,,,
baseline,25,83,83,82,83,100.0%,98.8%
opt1-batch,24,78,79,79,79,98.7%,100.0%
opt2-session-reuse,24,77,79,77,79,97.5%,97.5%
opt3-compact-json,23,71,73,70,73,97.3%,95.9%
opt4-idempotent,24,79,79,78,79,100.0%,98.7%
opt5-quiet,24,78,79,79,79,98.7%,100.0%
opt6-stable-output,23,74,76,76,76,97.4%,100.0%
